In [13]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

加载3Dmesh

In [14]:
import trimesh
mesh=trimesh.load("test.obj")
vertices=mesh.vertices
edges=mesh.edges
faces=mesh.faces

In [15]:
print(edges.shape)

(52242, 2)


In [16]:
vertices_tensor=torch.tensor(vertices,dtype=torch.float)
edges_tensor=torch.tensor(edges,dtype=torch.long).t().contiguous()  # t是转置操作，contiguous是连续存储

In [17]:
print(vertices_tensor.shape)
print(edges_tensor.shape)

torch.Size([8763, 3])
torch.Size([2, 52242])


创建包含顶点和边信息的Data对象

In [18]:
data=Data(x=vertices_tensor,edge_index=edges_tensor)

In [ ]:
from torch_geometric.utils import to_networkx
import networkx as nx
import matplotlib.pyplot as plt
G=to_networkx(data)
nx.draw(G)
plt.show()

In [22]:
class MeshGCN(torch.nn.Module):
    def __init__(self):
        super(MeshGCN,self).__init__()
        self.conv1=GCNConv(3,16)
        self.conv2=GCNConv(16,100)
    def forward(self,data):
        x,edge_index=data.x,data.edge_index
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=self.conv2(x,edge_index)
        return x

In [23]:
model=MeshGCN()
output=model(data)
print(type(output))

<class 'torch.Tensor'>


In [24]:
print(output.shape)

torch.Size([8763, 100])


使用PyTorch Geometric 的 DataLoader 可以自动处理变长图数据，无需手动对齐数据

In [27]:
import os
dir_path="./obj/obj_part1"
file_list=os.listdir(dir_path)

In [38]:
vertices_list=[]
edges_list=[]

for file in file_list:
    file_path=os.path.join(dir_path,file)
    mesh=trimesh.load(file_path)
    vertices_tensor=torch.tensor(mesh.vertices,dtype=torch.float)
    edges_tensor=torch.tensor(mesh.edges,dtype=torch.long).t().contiguous()
    print("vertices_tensor.shape:",vertices_tensor.shape)
    print("edges_tensor.shape:",edges_tensor.shape)
    vertices_list.append(vertices_tensor)
    edges_list.append(edges_tensor)

vertices_tensor.shape: torch.Size([10475, 3])
edges_tensor.shape: torch.Size([2, 62724])
vertices_tensor.shape: torch.Size([10475, 3])
edges_tensor.shape: torch.Size([2, 62724])
vertices_tensor.shape: torch.Size([10475, 3])
edges_tensor.shape: torch.Size([2, 62724])
vertices_tensor.shape: torch.Size([10475, 3])
edges_tensor.shape: torch.Size([2, 62724])


In [41]:
print(len(vertices_list))

4
Data(x=[10475, 3], edge_index=[2, 62724])


In [42]:
data_list=[]

for i in range(len(vertices_list)):
    data=Data(x=vertices_list[i],edge_index=edges_list[i])
    data_list.append(data)

In [43]:
from torch_geometric.loader import DataLoader

loader=DataLoader(data_list,batch_size=2,shuffle=True)

In [44]:
batch=next(iter(loader))
print(type(batch))

<class 'torch_geometric.data.batch.DataBatch'>
